# Occular-OCR — usage examples

Two scenarios: a **two-line quickstart**, and **full control** over every setting.

> Install: `pip install occular-ocr` (no compiler, no CUDA). Model weights download automatically
> from the Hugging Face Hub on first use and are cached locally.


## 1. Quickstart — two lines


In [ ]:
from ocr_skel import ocr

text = ocr("sample_page.png")     # an image path, or a ".pdf"
print(text)


Line-level output with coordinates and confidence:


In [ ]:
from ocr_skel import ocr_detailed

for line in ocr_detailed("sample_page.png"):
    print(f'{line["confidence"]:.2f}  {line["text"]}')


## 2. Full control — every setting

See all defaults at a glance:


In [ ]:
from ocr_skel import Settings
print(Settings())


### All settings

| Setting | Default | What it does |
|---|---|---|
| `num_threads` | `None` | CPU threads for inference. `None` → `min(cores, 4)`. |
| `gpu` | `False` | Run on GPU/CUDA. Requires `pip install occular-ocr[gpu]`; otherwise falls back to CPU. |
| `deskew` | `True` | Auto-correct skewed/rotated scans before detection. |
| `lm` | `True` | Beam search + language model (best quality). `False` → fast greedy decoding, skips the LM download. |
| `reading_order` | `False` | Order lines for multi-column layouts (downloads a small model, see below). |
| `detector` | `None` | Explicit detector name. `None` → default. |
| `recognizer` | `None` | Explicit recognizer name. `None` → default. |

PDF-only options (on `process_pdf`):

| Option | Default | What it does |
|---|---|---|
| `dpi` | `200` | Render resolution for scanned PDFs. Raise to `300` for small print. |
| `force_ocr` | `False` | OCR even PDFs that already contain a text layer. |
| `workers` | `None` | Parallel pages. `None` → auto `min(cores, 4)`; `1` → sequential. |


### Build a pipeline with all settings


In [ ]:
from ocr_skel import OCRPipeline, Settings

settings = Settings(
    num_threads=8,        # CPU threads (None -> min(cores, 4))
    gpu=False,            # True requires occular-ocr[gpu]
    deskew=True,          # auto-correct skewed scans
    lm=True,              # beam + language model; False -> greedy (faster, no LM download)
    reading_order=False,  # multi-column reading order (needs the optional model)
    detector=None,        # None = default detector
    recognizer=None,      # None = default recognizer
)

pipe = OCRPipeline(settings)

result = pipe.process_image("sample_page.png")   # -> list of {quad, text, confidence}
print(len(result), "lines")
print(result[0])


### PDF: render DPI + parallel workers


In [ ]:
pages = pipe.process_pdf(
    "document.pdf",
    dpi=300,          # render resolution (default 200)
    force_ocr=False,  # True: OCR even PDFs that already have a text layer
    workers=4,        # parallel pages (None = auto; 1 = sequential)
)
for i, page in enumerate(pages):
    print(f"page {i + 1}: {len(page['results'])} lines")


### One-liners: CPU/GPU · greedy/LM · deskew


In [ ]:
from ocr_skel import ocr

ocr("sample_page.png")                 # CPU (default), full quality
ocr("sample_page.png", gpu=True)       # GPU (requires occular-ocr[gpu])
ocr("sample_page.png", lm=False)       # fast greedy mode, no LM download
ocr("sample_page.png", deskew=False)   # skip deskew
ocr("sample_page.png", num_threads=2)  # limit CPU threads


### Reading order (multi-column) — optional model

Off by default; the model downloads once from the Hub.


In [ ]:
from ocr_skel import download_reading_order, model_info, OCRPipeline, Settings

download_reading_order()                       # one-time download
pipe = OCRPipeline(Settings(reading_order=True))

model_info()                                   # show which weights are present locally


### Folder → text files (command line)

Process every image/PDF in a folder and write `.txt` next to each:


In [ ]:
!python -m ocr_skel.cli  ./scans  ./out  --workers 4  --dpi 300
#   --gpu           run on GPU/CUDA
#   --dpi 300       PDF render resolution (default 200)
#   --force-ocr     OCR even vector PDFs
#   --workers 4     parallel workers
#   --out file.json save structured results to JSON
